# Sannidhi Edge TCN — CSI-Bench FallDetection

Self-contained Kaggle notebook. Attach only the CSI-Bench dataset. The loader uses only `/FallDetection/sub_Human/.../act_<activity>/.../*.h5`; `act_Fall` is positive and every other activity in FallDetection is negative. This is research validation, not ESP32 or clinical validation.

In [ ]:
!pip install -q h5py onnx onnxruntime onnxscript
import json, random
from pathlib import Path
import numpy as np, h5py, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
SEED=42; EPOCHS=30; MAX_EACH=800; T=32; CROP_FRAMES=400; TRAIN_AMPLITUDE_CHANNELS=116; N_CROPS=3; OUT=Path('/kaggle/working')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required for this notebook. Enable Kaggle GPU accelerator.')
device=torch.device('cuda'); torch.backends.cudnn.benchmark=True
print('GPU:',torch.cuda.get_device_name(0),'VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1e9,1))
DATA_ROOT=Path('/kaggle/input/datasets/guozhenjennzhu/csi-bench/FallDetection')
if not DATA_ROOT.exists():
    roots=[p for p in Path('/kaggle/input').iterdir() if p.is_dir() and p.name.lower()=='datasets']
    candidates=[p for root in roots for p in root.rglob('FallDetection') if p.is_dir()]
    DATA_ROOT=candidates[0] if candidates else None
if DATA_ROOT is None or not DATA_ROOT.exists(): raise RuntimeError('CSI-Bench FallDetection path not found; inspect /kaggle/input.')
files=list(DATA_ROOT.glob('sub_Human/**/*.h5'))
if not files: raise RuntimeError('No FallDetection H5 files found.')
def activity_label(p):
    acts=[x for x in p.parts if x.lower().startswith('act_')]
    if not acts: return -1
    return int(acts[-1].lower().replace('-','_') in {'act_fall','act_falling','act_fallen','act_fall_down'})
falls=[p for p in files if activity_label(p)==1]; nonfalls=[p for p in files if activity_label(p)==0]
if not falls or not nonfalls: raise RuntimeError(f'Need both classes; found falls={len(falls)}, nonfalls={len(nonfalls)}.')
rng=np.random.default_rng(SEED); rng.shuffle(falls); rng.shuffle(nonfalls)
falls,nonfalls=falls[:MAX_EACH],nonfalls[:MAX_EACH]; cut_f=int(.8*len(falls)); cut_n=int(.8*len(nonfalls))
train_paths=[(p,1) for p in falls[:cut_f]]+[(p,0) for p in nonfalls[:cut_n]]; valid_paths=[(p,1) for p in falls[cut_f:]]+[(p,0) for p in nonfalls[cut_n:]]
rng.shuffle(train_paths); rng.shuffle(valid_paths); paths=train_paths+valid_paths
print('FallDetection files:',len(files),'selected:',len(paths),'train files:',len(train_paths),'validation files:',len(valid_paths))
print('fall example:',falls[0]); print('non-fall example:',nonfalls[0])

## Feature extraction and low-latency causal model

In [ ]:
def read_amp(p):
    with h5py.File(p,'r') as f: x=f['CSI_amps'][:]
    if x.ndim==3: x=np.squeeze(x)
    if x.ndim!=2: raise ValueError('Expected 2D CSI after squeeze, got '+str(x.shape))
    if x.shape[0] < x.shape[1]: x=x.T
    return x.astype('float32')
TARGET_FEATURES=None
ALIGNMENT_COUNTS={'padded':0,'truncated':0,'matched':0}
def align_channels(x):
    global ALIGNMENT_COUNTS
    if x.shape[1] > TARGET_FEATURES:
        ALIGNMENT_COUNTS['truncated'] += 1; return x[:, :TARGET_FEATURES]
    if x.shape[1] < TARGET_FEATURES:
        ALIGNMENT_COUNTS['padded'] += 1; return np.pad(x, ((0,0),(0,TARGET_FEATURES-x.shape[1])), mode='constant')
    ALIGNMENT_COUNTS['matched'] += 1; return x
def feature_crop(x):
    q=np.isfinite(x).all(1,keepdims=True).astype('float32'); x=np.nan_to_num(x,nan=0.,posinf=0.,neginf=0.)
    x=align_channels(x)
    old=np.linspace(0,1,len(x)); new=np.linspace(0,1,T); q=np.interp(new,old,q[:,0])[:,None]; x=np.stack([np.interp(new,old,x[:,k]) for k in range(x.shape[1])],1)
    med=np.median(x,0,keepdims=True); mad=np.median(np.abs(x-med),0,keepdims=True); amp=np.clip((x-med)/(1.4826*mad+1e-5),-8,8)
    d=np.vstack([np.zeros((1,x.shape[1]),'float32'),np.diff(x,axis=0)]); med=np.median(d,0,keepdims=True); mad=np.median(np.abs(d-med),0,keepdims=True); d=np.clip((d-med)/(1.4826*mad+1e-5),-8,8)
    e=np.sqrt(np.mean(d*d,1,keepdims=True))
    return np.concatenate([amp,d,e,q],1).astype('float32')
def build_cache(items, augment=False):
    global TARGET_FEATURES
    if TARGET_FEATURES is None: TARGET_FEATURES=TRAIN_AMPLITUDE_CHANNELS; print('CSI amplitude channels used for ESP32 contract:',TARGET_FEATURES)
    xs=[]; ys=[]
    for i,(p,y) in enumerate(items):
        raw=read_amp(p); max_start=max(0,len(raw)-CROP_FRAMES)
        starts=[0,max_start//2,max_start] if augment else [max_start//2]
        for start in sorted(set(starts)):
            crop=raw[start:start+CROP_FRAMES]
            if len(crop)<CROP_FRAMES: crop=np.pad(crop,((0,CROP_FRAMES-len(crop)),(0,0)),mode='edge')
            z=feature_crop(crop); xs.append(z); ys.append([y,float(np.mean(z[:,-2])>.15),float(np.isfinite(z).all())])
        if (i+1)%200==0: print('cached',i+1,'/',len(items))
    return np.stack(xs).astype('float32'),np.asarray(ys,dtype='float32')
print('Precomputing CPU features once; training batches will stay on GPU...')
Xtr,Ytr=build_cache(train_paths,augment=True); Xva,Yva=build_cache(valid_paths,augment=False)
print('cached train:',Xtr.shape,'validation:',Xva.shape,'channel alignment:',ALIGNMENT_COUNTS)

## Training, validation, and export

In [ ]:
class Block(nn.Module):
    def __init__(self,c,d):
        super().__init__(); self.pad=2*d; self.dw=nn.Conv1d(c,c,3,dilation=d,groups=c,bias=False); self.pw=nn.Conv1d(c,c,1); self.bn=nn.BatchNorm1d(c)
    def forward(self,x):
        r=x; x=nn.functional.pad(x,(self.pad,0)); return r+nn.functional.gelu(self.bn(self.pw(self.dw(x))))
class EdgeTCN(nn.Module):
    def __init__(self,n,c=48,drop=.08):
        super().__init__(); self.inp=nn.Conv1d(n,c,1); self.blocks=nn.Sequential(*[Block(c,d) for d in (1,2,4,8)]); self.score=nn.Conv1d(c,1,1); h=max(6,c//8)
        self.gate=nn.Sequential(nn.Linear(c,h),nn.GELU(),nn.Linear(h,c),nn.Sigmoid()); self.head=nn.Sequential(nn.LayerNorm(c*3),nn.Linear(c*3,32),nn.GELU(),nn.Dropout(drop),nn.Linear(32,3))
    def forward(self,x):
        x=self.blocks(self.inp(x.transpose(1,2))); a=(x*torch.softmax(self.score(x),-1)).sum(-1); m=x.amax(-1); l=x[:,:,-1]
        f=torch.cat([a,m,l],1); g=torch.cat([self.gate(a),self.gate(m),self.gate(l)],1); return self.head(f*g)
pos=max(1.,float((Ytr[:,0]==0).sum()))/max(1.,float((Ytr[:,0]==1).sum()))
train_ds=TensorDataset(torch.from_numpy(Xtr).to(device),torch.from_numpy(Ytr).to(device)); val_ds=TensorDataset(torch.from_numpy(Xva).to(device),torch.from_numpy(Yva).to(device))
tr=DataLoader(train_ds,128,shuffle=True,num_workers=0); va=DataLoader(val_ds,256,shuffle=False,num_workers=0); n=Xtr.shape[2]
def collect(model):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in va: ys.append(y[:,0].cpu()); ps.append(torch.sigmoid(model(x))[:,0].cpu())
    return torch.cat(ys).numpy(),torch.cat(ps).numpy()
def score(model):
    y,p=collect(model); best={'f1':-1}
    for t in np.linspace(.1,.9,81):
        z=p>=t; tp=((z)&(y==1)).sum(); fp=((z)&(y==0)).sum(); fn=((~z)&(y==1)).sum(); pr=tp/max(1,tp+fp); re=tp/max(1,tp+fn); f=2*pr*re/max(1e-9,pr+re)
        if f>best['f1']: best={'precision':float(pr),'recall':float(re),'f1':float(f),'threshold':float(t)}
    return best
def train(model,epochs,teacher=None):
    model.to(device); bce=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos,device=device)); opt=torch.optim.AdamW(model.parameters(),6e-4,weight_decay=3e-4); best=-1.; stale=0
    for ep in range(epochs):
        model.train(); total=0.
        for x,y in tr:
            z=model(x); loss=bce(z[:,0],y[:,0])+.1*bce(z[:,1],y[:,1])+.1*bce(z[:,2],y[:,2])
            if teacher is not None:
                with torch.no_grad(): tz=teacher(x)
                temp=2.; loss=loss+.25*nn.functional.kl_div(nn.functional.log_softmax(z/temp,1),nn.functional.softmax(tz/temp,1),reduction='batchmean')*temp*temp
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); total+=loss.item()
        m=score(model); print('epoch=%02d loss=%.4f precision=%.3f recall=%.3f f1=%.3f threshold=%.2f'%(ep+1,total/max(1,len(tr)),m['precision'],m['recall'],m['f1'],m['threshold']))
        if m['f1']>best: best=m['f1']; best_m=m.copy(); best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=7: break
    model.load_state_dict(best_state); return best_m
teacher=EdgeTCN(n,96,.10); student=EdgeTCN(n,48,.08); distilled=EdgeTCN(n,48,.08)
print('features:',n,'teacher_params:',sum(p.numel() for p in teacher.parameters()),'student_params:',sum(p.numel() for p in student.parameters()))
print('TRAIN TEACHER'); teacher_m=train(teacher,20)
teacher.eval(); print('TRAIN SUPERVISED STUDENT'); student_m=train(student,30)
print('TRAIN DISTILLED STUDENT'); distilled_m=train(distilled,30,teacher)
results={'teacher':teacher_m,'student_supervised':student_m,'student_distilled':distilled_m}; print(json.dumps(results,indent=2))
chosen=distilled if distilled_m['f1']>=student_m['f1'] else student; chosen_m=distilled_m if distilled_m['f1']>=student_m['f1'] else student_m
torch.save({'model_state_dict':chosen.state_dict(),'input_features':n,'window_frames':T,'crop_frames_at_100hz':CROP_FRAMES,'metrics':chosen_m,'candidates':results,'seed':SEED},OUT/'edge_tcn_candidate.pt')
json.dump({'chosen':'student_distilled' if chosen is distilled else 'student_supervised','candidates':results,'teacher_not_deployed':True,'dataset':'CSI-Bench/FallDetection','esp32_validation':'not_assessed','seed':SEED},open(OUT/'edge_tcn_candidate_metrics.json','w'),indent=2)
cpu_model=chosen.to('cpu').eval()
try:
    torch.onnx.export(cpu_model,torch.zeros(1,T,n),OUT/'edge_tcn_candidate.onnx',input_names=['features'],output_names=['logits'],dynamic_axes={'features':{0:'batch'},'logits':{0:'batch'}},opset_version=18)
    import onnxruntime as ort,time
    sess=ort.InferenceSession(str(OUT/'edge_tcn_candidate.onnx'),providers=['CPUExecutionProvider']); sample=np.zeros((1,T,n),dtype='float32'); a=cpu_model(torch.from_numpy(sample)).detach().numpy(); b=sess.run(['logits'],{'features':sample})[0]; times=[]
    for _ in range(30):
        t0=time.perf_counter(); sess.run(['logits'],{'features':sample}); times.append((time.perf_counter()-t0)*1000)
    print('onnx_max_abs_difference:',float(np.max(np.abs(a-b))),'cpu_latency_ms_median:',float(np.median(times)),'p95:',float(np.percentile(times,95)))
except Exception as e: print('onnx_validation_failed:',type(e).__name__,e)
print('artifacts:',[(p.name,p.exists(),p.stat().st_size if p.exists() else 0) for p in (OUT/'edge_tcn_candidate.pt',OUT/'edge_tcn_candidate.onnx',OUT/'edge_tcn_candidate_metrics.json')])

## SOTA-lite benchmark: PatchTST-style transformer

In [ ]:
# Run this cell after the data-cache cell. It replaces the PatchTST benchmark.
# InceptionTime-style multi-scale convolutions preserve short fall transients and are more data-efficient than the transformer here.
class InceptionBlock(nn.Module):
    def __init__(self,cin,hidden=24,drop=.08):
        super().__init__(); self.reduce=nn.Conv1d(cin,hidden,1,bias=False); self.branches=nn.ModuleList([nn.Conv1d(hidden,hidden,k,padding=k//2,bias=False) for k in (3,5,9)])
        self.pool=nn.Sequential(nn.MaxPool1d(3,1,1),nn.Conv1d(cin,hidden,1,bias=False)); self.mix=nn.Sequential(nn.BatchNorm1d(hidden*4),nn.GELU(),nn.Conv1d(hidden*4,64,1,bias=False),nn.BatchNorm1d(64),nn.GELU(),nn.Dropout(drop)); self.skip=nn.Identity() if cin==64 else nn.Conv1d(cin,64,1,bias=False)
    def forward(self,x):
        z=self.reduce(x); z=torch.cat([b(z) for b in self.branches]+[self.pool(x)],1); return self.mix(z)+self.skip(x)
class InceptionTimeLite(nn.Module):
    def __init__(self,n,drop=.08):
        super().__init__(); self.stem=nn.Conv1d(n,32,1,bias=False); self.body=nn.Sequential(InceptionBlock(32,24,drop),InceptionBlock(64,24,drop)); self.score=nn.Conv1d(64,1,1); self.head=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,32),nn.GELU(),nn.Dropout(drop),nn.Linear(32,3))
    def forward(self,x):
        z=self.body(self.stem(x.transpose(1,2))); a=(z*torch.softmax(self.score(z),-1)).sum(-1); m=z.amax(-1); return self.head(torch.cat([a,m],1))
pos=max(1.,float((Ytr[:,0]==0).sum()))/max(1.,float((Ytr[:,0]==1).sum()))
train_ds=TensorDataset(torch.from_numpy(Xtr).to(device),torch.from_numpy(Ytr).to(device)); val_ds=TensorDataset(torch.from_numpy(Xva).to(device),torch.from_numpy(Yva).to(device))
tr=DataLoader(train_ds,128,shuffle=True,num_workers=0); va=DataLoader(val_ds,256,shuffle=False,num_workers=0)
def score(model):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in va: ys.append(y[:,0].cpu()); ps.append(torch.sigmoid(model(x))[:,0].cpu())
    y=np.concatenate([a.numpy() for a in ys]); p=np.concatenate([a.numpy() for a in ps]); best={'f1':-1}
    for t in np.linspace(.1,.9,81):
        z=p>=t; tp=((z)&(y==1)).sum(); fp=((z)&(y==0)).sum(); fn=((~z)&(y==1)).sum(); pr=tp/max(1,tp+fp); re=tp/max(1,tp+fn); f=2*pr*re/max(1e-9,pr+re)
        if f>best['f1']: best={'precision':float(pr),'recall':float(re),'f1':float(f),'threshold':float(t)}
    return best
def stable_fit(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); m=PatchTSTLite(Xtr.shape[2]).to(device)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4,weight_decay=1e-3); loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos,device=device)); best=-1.; stale=0
    for ep in range(30):
        m.train(); total=0.
        for x,y in tr:
            with torch.autocast(device_type='cuda',dtype=torch.float16): z=m(x); loss=loss_fn(z[:,0],y[:,0])+.1*loss_fn(z[:,1],y[:,1])+.1*loss_fn(z[:,2],y[:,2])
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(),.75); opt.step(); total+=loss.item()
        cur=score(m); print('seed=%d epoch=%02d loss=%.4f f1=%.3f precision=%.3f recall=%.3f threshold=%.2f'%(seed,ep+1,total/max(1,len(tr)),cur['f1'],cur['precision'],cur['recall'],cur['threshold']))
        if cur['f1']>best: best=cur['f1']; best_m=cur.copy(); best_state={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=6: break
    m.load_state_dict(best_state); return m,best_m
runs=[stable_fit(s) for s in (11,22,33)]; chosen,chosen_m=max(runs,key=lambda r:r[1]['f1']); print('PATCHTST_RESULTS',json.dumps([m for _,m in runs],indent=2))
torch.save({'model_state_dict':chosen.state_dict(),'input_features':Xtr.shape[2],'window_frames':T,'metrics':chosen_m,'architecture':'PatchTSTLite','seeds':[11,22,33]},OUT/'edge_sota_candidate.pt')
json.dump({'architecture':'PatchTSTLite','input_features':int(Xtr.shape[2]),'seeds':[11,22,33],'selected_metrics':chosen_m,'all_runs':[m for _,m in runs],'validation_only':True},open(OUT/'edge_sota_candidate_metrics.json','w'),indent=2)
cpu_model=chosen.to('cpu').eval(); onnx_path=OUT/'edge_sota_candidate.onnx'
torch.onnx.export(cpu_model,torch.zeros(1,T,Xtr.shape[2]),onnx_path,input_names=['features'],output_names=['logits'],dynamic_axes={'features':{0:'batch'},'logits':{0:'batch'}},opset_version=18)
print('SOTA candidate saved:',onnx_path,'params:',sum(p.numel() for p in chosen.parameters()),'metrics:',chosen_m)